# Поиск по всем полевым фото

Тот же запуск, что для одного снимка:

- энкодер `weights/finetuned_v4/encoder.onnx`
- индекс `weights/finetuned_v4/books_index.npz`
- эталон `../artifacts_clean/books`
- top-3
- детектор из `configs/detection/detection_config.yaml`

На каждое фото: снимок с рамкой, кроп и три ближайших клейма. Если детекций несколько, строка на каждую.

In [ ]:
from __future__ import annotations

import importlib
import io
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Image, display
from PIL import Image as PILImage
from PIL import ImageDraw, ImageFont
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src" / "pipeline.py").is_file():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Запускайте ноутбук из репозитория CeramicNet")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.detection.detector import StampDetector
from src.paths import CONFIGS_DIR
from src.pipeline import CeramicPipeline
from src.preprocessing.crop import crop_stamp
from src.preprocessing.image import read_bgr
from src.retrieval.encoder import StampEncoder
from src.retrieval.index import StampIndex
from src.utils import load_yaml

ENCODER = ROOT / "weights/finetuned_v4/encoder.onnx"
INDEX = ROOT / "weights/finetuned_v4/books_index.npz"
BOOKS = (ROOT.parent / "artifacts_clean" / "books").resolve()
TOP_K = 10
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg"}
SIDE = 220
MATCH = 160
GAP = 6


def find_stamps(root: Path) -> Path:
    for base in (root, root.parent):
        if not base.is_dir():
            continue
        for child in sorted(base.iterdir(), key=lambda path: path.name):
            stamps = child / "stamps"
            if stamps.is_dir() and any(stamps.glob("*.png")):
                return stamps
    raise FileNotFoundError("Папка stamps не найдена рядом с репозиторием")


def list_images(folder: Path) -> list[Path]:
    return sorted(
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )


for path, title in ((ENCODER, "энкодер"), (INDEX, "индекс"), (BOOKS, "эталон")):
    if not path.exists():
        raise FileNotFoundError(f"Не найден {title}: {path}")

ret_cfg = load_yaml(CONFIGS_DIR / "retrieval" / "retrieval_config.yaml")
det_cfg = load_yaml(CONFIGS_DIR / "detection" / "detection_config.yaml")

encoder = StampEncoder.from_mapping(ret_cfg["encoder"], weights=ENCODER)
index = StampIndex.load(INDEX, books_dir=BOOKS)

module_path, class_name = det_cfg["processor"].rsplit(".", 1)
processor_cls = getattr(importlib.import_module(module_path), class_name)
processor = processor_cls(
    imgsz=det_cfg.get("imgsz", 640),
    conf_threshold=det_cfg.get("conf_threshold", 0.7),
    iou_threshold=det_cfg.get("iou_threshold", 0.5),
)
detector = StampDetector(model_path=ROOT / det_cfg["weights"], processor=processor)
pipeline = CeramicPipeline(
    encoder=encoder,
    index=index,
    detector=detector,
    conf_threshold=float(det_cfg.get("conf_threshold", 0.75)),
    top_k=TOP_K,
)

STAMPS_DIR = find_stamps(ROOT)
PHOTOS = list_images(STAMPS_DIR)
if not PHOTOS:
    raise FileNotFoundError(f"В папке нет изображений: {STAMPS_DIR}")

try:
    RANK_FONT = ImageFont.truetype("/System/Library/Fonts/Supplemental/Arial.ttf", 16)
except OSError:
    RANK_FONT = ImageFont.load_default()

print(f"Фото в папке: {len(PHOTOS)}")
print(f"Папка: {STAMPS_DIR}")
print(f"Энкодер: {ENCODER.name}")
print(f"Индекс: {INDEX.name}, эталонов {len(index.records)}")
print(f"Детектор: {Path(det_cfg['weights']).name}, порог {pipeline.conf_threshold}, top-{TOP_K}")


In [ ]:
def fit_square(image: PILImage.Image, size: int, fill: tuple[int, int, int]) -> PILImage.Image:
    image = image.copy()
    image.thumbnail((size, size))
    canvas = PILImage.new("RGB", (size, size), fill)
    canvas.paste(image, ((size - image.width) // 2, (size - image.height) // 2))
    return canvas


def photo_tile(image_bgr, box) -> PILImage.Image:
    rgb = PILImage.fromarray(image_bgr[:, :, ::-1])
    if box is not None:
        width = max(rgb.size)
        ImageDraw.Draw(rgb).rectangle(
            tuple(int(v) for v in box),
            outline=(210, 35, 35),
            width=max(3, width // 90),
        )
    return fit_square(rgb, SIDE, (244, 244, 244))


def book_tile(path: str, rank: int) -> PILImage.Image:
    canvas = PILImage.new("RGB", (MATCH, MATCH), (255, 255, 255))
    if path and Path(path).is_file():
        image = PILImage.open(path).convert("RGB")
        image.thumbnail((MATCH - 8, MATCH - 8))
        canvas.paste(image, ((MATCH - image.width) // 2, (MATCH - image.height) // 2))
    draw = ImageDraw.Draw(canvas)
    badge = 20 if rank < 10 else 28
    draw.rectangle((0, 0, badge, 22), fill=(255, 255, 255))
    draw.text((3, 2), str(rank), fill=(20, 20, 20), font=RANK_FONT)
    return canvas


def match_line(item: dict) -> str:
    factory = item.get("factory") or "—"
    catalog = item.get("catalog_no") or ""
    number = f" №{catalog}" if catalog else ""
    return f"{item['score']:.3f}  {item['mark_id']}  {item.get('book') or ''}{number}  {factory}"


def sheet_for(image_bgr, box, matches: list[dict]) -> PILImage.Image:
    photo = photo_tile(image_bgr, box if box is not None else None)
    if box is None:
        crop = PILImage.new("RGB", (SIDE, SIDE), (220, 220, 220))
    else:
        crop_bgr = crop_stamp(image_bgr, box, target_size=pipeline.target_size)
        crop = fit_square(PILImage.fromarray(crop_bgr[:, :, ::-1]), SIDE, (244, 244, 244))
    width = SIDE * 2 + GAP + TOP_K * MATCH + TOP_K * GAP
    sheet = PILImage.new("RGB", (width, SIDE), (244, 244, 244))
    sheet.paste(photo, (0, (SIDE - photo.height) // 2))
    sheet.paste(crop, (SIDE + GAP, (SIDE - crop.height) // 2))
    origin = SIDE * 2 + GAP * 2
    for item in matches[:TOP_K]:
        tile = book_tile(item["image_path"], item["rank"])
        y = (SIDE - tile.height) // 2
        sheet.paste(tile, (origin, y))
        origin += MATCH + GAP
    return sheet


def show_sheet(sheet: PILImage.Image) -> None:
    buffer = io.BytesIO()
    sheet.save(buffer, format="JPEG", quality=85)
    display(Image(data=buffer.getvalue()))


rows = []
cards = []
for path in tqdm(PHOTOS, desc="фото"):
    result = pipeline.predict(path, top_k=TOP_K, detect=True)
    detections = result["detections"]
    image_bgr = read_bgr(path)
    if not detections:
        rows.append({
            "файл": path.name,
            "детекция": 0,
            "conf": None,
            "top1": None,
            "mark_id": "",
            "книга": "",
            "завод": "",
        })
        cards.append({"name": path.name, "image": image_bgr, "box": None, "score": None, "matches": [], "i": 0, "n": 0})
        continue
    for number, detection in enumerate(detections, start=1):
        matches = detection["matches"]
        first = matches[0] if matches else None
        rows.append({
            "файл": path.name,
            "детекция": number,
            "conf": detection["score"],
            "top1": None if first is None else first["score"],
            "mark_id": "" if first is None else first["mark_id"],
            "книга": "" if first is None else first.get("book") or "",
            "завод": "" if first is None else first.get("factory") or "",
        })
        cards.append({
            "name": path.name,
            "image": image_bgr,
            "box": detection["box"],
            "score": detection["score"],
            "matches": matches,
            "i": number,
            "n": len(detections),
        })

found = sum(1 for card in cards if card["score"] is not None)
missed = sum(1 for card in cards if card["n"] == 0)
print(f"Строк: {len(cards)}. С клеймом: {found}. Без детекции: {missed}.")

pd.set_option("display.max_rows", 1000)
pd.set_option("display.max_colwidth", 80)
table = pd.DataFrame(rows)
display(table)

for card in cards:
    if card["score"] is None:
        caption = f"<b>{card['name']}</b><br>клеймо не найдено"
    else:
        extra = ""
        if card["n"] > 1:
            extra = f" · детекция {card['i']} из {card['n']}"
        lines = "<br>".join(
            f"{item['rank']}. {match_line(item)}" for item in card["matches"]
        )
        caption = f"<b>{card['name']}</b> · conf {card['score']:.3f}{extra}<br>{lines}"
    display(HTML(caption))
    show_sheet(sheet_for(card["image"], card["box"], card["matches"]))
